In [131]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
from datetime import datetime

In [2]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [3]:
from selenium.webdriver.chrome.service import Service
s = Service(r"/usr/bin/chromedriver")


# Observations

In [115]:
import json
import copy
driver = webdriver.Chrome(service=s)
driver.get("https://flatheadavalanche.org/avalanche-forecast/#/archive")
html = driver.page_source

In [114]:
import json
import copy
import time
weather_df = pd.DataFrame()
forms = []

In [112]:
year = 2024

In [128]:
forms[-1]

{'author': 'Rob Millspaugh',
 'date': '2023-12-22',
 'zone': 'Flathead Range & Glacier NP',
 'problems': [{'problem_type': ' Wet Loose',
   'problem_number': 1,
   'likelihood': 'possible',
   'size': ['Small (D1)', ''],
   'aspect_elevation': {'upper': ['north',
     'northeast',
     'east',
     'southeast',
     'south',
     'southwest',
     'west',
     'northwest'],
    'middle': ['north',
     'northeast',
     'east',
     'southeast',
     'south',
     'southwest',
     'west',
     'northwest'],
    'lower': []},
   'discussion': 'You can trigger sluffs of dense, wet snow on slopes steeper than about 38 degrees. These can be dangerous on slopes above terrain traps and will run farther slopes without low-angle benches that can slow moving debris. Rollerballs that fan out from your machine or boards are a clear sign that the surface snow is wet. Step back to lower-angled terrain if you see these conditions. This hazard may be more widespread on shaded slopes where the surfac

In [ ]:
n = 0
while(True):
    driver.implicitly_wait(20)
    html = driver.page_source
    soup = BeautifulSoup(html, 'html.parser')
    new_form, new_weather = parse_zone_forecast_html(html)
    if new_form in forms:
        break
    else:
        forms.append(copy.deepcopy(new_form))
        weather_df = pd.concat([weather_df, new_weather])
    button = driver.find_element(By.CSS_SELECTOR, '.nac-html-button.nac-btn.nac-btn-light.nac-btn-lg.has-tooltip')
    button.click()
    time.sleep(20)
    n+=1
    print(len(forms))
    print(new_form)
    print('\n')
    pd.DataFrame(forms).to_csv(f'/home/christine/Jupyter/snow/data/fac_forecast_{int(year-1)}_{int(year)}.csv')
    weather_df.to_csv(f'/home/christine/Jupyter/snow/data/fac_weather_{int(year-1)}_{int(year)}.csv')


89
{'author': 'blase reardon', 'date': '2023-12-21', 'zone': 'Flathead Range & Glacier NP', 'problems': [{'problem_type': ' Wet Loose', 'problem_number': 1, 'likelihood': 'possible', 'size': ['Small (D1)', ''], 'aspect_elevation': {'upper': [], 'middle': ['north', 'northeast', 'east', 'southeast', 'south', 'southwest', 'west', 'northwest'], 'lower': []}, 'discussion': 'You can trigger sluffs of dense, wet snow on mid-elevation slopes steeper than about 38 degrees. These can be dangerous on slopes above terrain traps or slopes without low-angle benches that can slow moving debris. Rollerballs that fan out from your machine or boards are a reliable sign that the surface snow is wet. Step back to lower-angled terrain if you see these conditions. This hazard may be more widespread on shaded slopes where the surface snow is getting wet for the first time.\xa0'}], 'bottom_line': "It's the winter solstice, so of course the primary avalanche concern is... triggering wet snow avalanches on very

In [133]:
stations = {'Flathead Range & Glacier NP':'Shed 7','Swan':'Aeneas Ridge','Whitefish':'WMR Summit'}

def get_weather_table(soup, zone, date):
    forecast_table = soup.find('table', {'class': 'nac-html-table nac-table'})

    # Initialize an empty list to hold all rows of data
    data = []

    # Extract the header row
    headers = [header.get_text(strip=True) for header in forecast_table.find_all('th')]

    # Iterate through each row in the table body
    for row in forecast_table.find('tbody').find_all('tr'):
        cols = row.find_all('td')
        # Extract text from each cell in the row and add to the row data
        row_data = [col.get_text(strip=True) for col in cols]
        row_data = [r for r in row_data if '12hr' not in r]
        data.append(row_data)
    df = pd.DataFrame(data, columns=headers)

    # Adjust the DataFrame to have proper headers
#     new_header = ['Metric', 'Last 24 Hours', 'Today', 'Tonight', 'Tomorrow']
#     df.columns = new_header
    df['range'] = zone
 #   df['station'] = stations.get(zone)
    df['date'] = date
    return df

def get_author_date_zone(soup):
    author = soup.find('h6', string='Author').find_next_sibling(text=True).strip()
    zone_element = soup.find('h2', class_='nac-zone-title')
    zone = zone_element.text.strip()
    issued_date = soup.find('h6', string='Issued').find_next_sibling(text=True).strip()
    issued_date = datetime.strptime(issued_date, "%A, %B %d, %Y - %I:%M%p")
    date_formatted = issued_date.strftime('%Y-%m-%d')
    return {'author' : author, 'zone': zone, 'date': date_formatted}

def get_forecast(soup):
    today_forecast_container = soup.find('div', class_='nac-dangerToday')

    forecast = {'upper': '', 'middle': '', 'lower': ''}
    elevation_blocks = today_forecast_container.find_all('div', class_='nac-elevationBlock')
    for block in elevation_blocks:
        elevation_label = block.find('span', class_='nac-elevationLabel').text.strip().lower()
        danger_label = block.find('span', class_='nac-dangerLabel').text.strip().split(' - ')[1]

        if 'upper' in elevation_label:
            forecast['upper'] = danger_label
        elif 'mid' in elevation_label:
            forecast['middle'] = danger_label
        elif 'low' in elevation_label:
            forecast['lower'] = danger_label
        return forecast

def get_bottom_line(soup):
    return soup.find('div', class_ = 'nac-bottomLine-text nac-tinymce').text

def get_discussion(soup):
    return soup.find('h2', text='Forecast Discussion').find_next_sibling().text

def parse_problems(soup):
    parsed_json_list = []
    
    # Find all problem blocks
    problem_blocks = soup.find_all('h3', class_='nac-html-h3')
    problem_blocks = list(set([p for p in problem_blocks if 'Problem' in p.text]))
    
    for problem_h3 in problem_blocks:
        problem_type_text = problem_h3.get_text(strip=True)
        problem_number, problem_type = problem_type_text.split(':')
        problem_number = int(problem_number.replace("Problem #", "").strip())

        # Assuming the structure is consistent for each problem block
        problem_block = problem_h3.find_next_sibling('div', class_='nac-infoGraphics nac-row')
        
        # Likelihood
        likelihood_label = problem_block.find('h5', text=' Likelihood ').find_next_sibling('div')
        likelihood = likelihood_label.find('div', class_='nac-label nac-active').get_text(strip=True).lower()

        # Size
        size_label = problem_block.find('h5', text=' Size ').find_next_sibling('div')
        size = [label.get_text(strip=True) for label in size_label.find_all('div', class_='nac-label nac-active')]

        # Aspect/Elevation
        aspect_elevation = {'upper': [], 'middle': [], 'lower': []}
        svg_paths = problem_block.find('div', class_='nac-rose').find_all('path', {'data-id': True})
        for path in svg_paths:
            aspect, elevation = path['data-id'].split()
            if 'rgb(200, 202, 206)' in path['style']:  # Check if the path is shaded
                aspect_elevation[elevation.lower()].append(aspect)

        # Discussion
        discussion_block = problem_h3.find_next_sibling('div', class_='nac-tinymce')
        if discussion_block:  # Check if the discussion block exists
            discussion_paragraphs = discussion_block.select('p')
            discussion = ' '.join(p.get_text() for p in discussion_paragraphs)
        else:
            discussion = ""

        # Combine into JSON structure
        parsed_json = {
            'problem_type': problem_type,
            'problem_number': problem_number,
            'likelihood': likelihood,
            'size': size,
            'aspect_elevation': aspect_elevation,
            'discussion': discussion,}
        

        parsed_json_list.append(parsed_json)
    return parsed_json_list

def parse_zone_forecast_html(html):
    soup = BeautifulSoup(html, 'html.parser')
    
    author_date_zone_info = get_author_date_zone(soup)
    
    problems = parse_problems(soup)
    
    bottom_line = get_bottom_line(soup)
    
    discussion = get_discussion(soup)
    
    weather_df = get_weather_table(soup, author_date_zone_info['zone'], author_date_zone_info['date'])

    # Assemble everything into a structured dictionary
    forecast_info = {
        'author': author_date_zone_info['author'],
        'date': author_date_zone_info['date'],
        'zone': author_date_zone_info['zone'],
        'problems': problems,
        'bottom_line': bottom_line,
        'discussion': discussion,
    }
    return forecast_info, weather_df

In [121]:
data = [['Temperature', '32F', '31-38F', '26-29F'], ['Cloud Cover', 'Overcast', 'Mostly Cloudy', 'Partly Cloudy'], ['Ridgeline Wind Speed', 'Light', 'Light', 'Light'], ['Wind Direction', 'W', 'SW', 'SW'], ['Snowfall', '12hr:0"', '24hr:0"', '0"', '0"'], ['Snow Water Equivalent', '12hr:0"', '24hr:0.09"', '0"', '0"']]

In [126]:
pd.DataFrame(data)

,0,1,2,3,4
0,Temperature,32F,31-38F,26-29F,None
1,Cloud Cover,Overcast,Mostly Cloudy,Partly Cloudy,None
2,Ridgeline Wind Speed,Light,Light,Light,None
3,Wind Direction,W,SW,SW,None
4,Snowfall,"12hr:0""","24hr:0""","0""","0"""
5,Snow Water Equivalent,"12hr:0""","24hr:0.09""","0""","0"""
